# 06 — Monitoring & évaluation

**Pré-requis :** avoir exécuté le notebook [01-fondamentaux-azure-paas.ipynb](01-fondamentaux-azure-paas.ipynb) (Web App + projet Foundry) et idéalement le [02-foundry-agent-playground.ipynb](02-foundry-agent-playground.ipynb) (agent IA). **Gardez** le Resource Group créé en 01.

**Objectif :** brancher de l'observabilité sur la Web App **et préparer l'évaluation** des agents Foundry.

Ce notebook répond à des questions du genre :
- combien de requêtes ma webapp reçoit-elle, et combien échouent ?
- quelles routes sont lentes ? quelles exceptions sont levées ?
- quelle est ma consommation de tokens IA, quelle est la latence de l'agent ?

À la fin de ce notebook, vous aurez :
- compris la stack **Azure Monitor / Log Analytics / Application Insights** ;
- créé un **Log Analytics Workspace** et une ressource **Application Insights** ;
- branché la connection string sur la Web App ;
- envoyé une trace de test depuis Python avec OpenTelemetry ;
- écrit vos premières requêtes **KQL** pour explorer les données.

> 🚧 **Évaluation des agents Foundry** (groundedness, relevance, safety…) : section à venir dans une prochaine itération de ce notebook.

## 1. Concepts — Observabilité sur Azure

### Les trois piliers de l'observabilité

| Pilier | Question répondue | Exemple |
|--------|-------------------|---------|
| **Logs** | *Que s'est-il passé ?* | « Exception NullReferenceException dans `/api/order` à 14:32 » |
| **Métriques** | *Combien ? À quelle vitesse ?* | « CPU à 78 %, 1 200 requêtes/min, latence p95 = 240 ms » |
| **Traces** | *Comment la requête s'est-elle propagée ?* | « `/api/order` → SQL `SELECT…` 80 ms → `POST /payments` 120 ms » |

### La stack Azure

```
                 ┌──────────────────────────────────────────┐
                 │           Azure Monitor                  │
                 │  (la marque / l'expérience unifiée)      │
                 └──────────────────────────────────────────┘
                                  │
        ┌─────────────────────────┴─────────────────────────┐
        │                                                   │
  Application Insights                       Métriques de plateforme
  (APM applicatif : requêtes, dépendances,   (CPU, mémoire, IOPS… par ressource)
   exceptions, traces, logs custom)
        │
        ▼  (workspace-based depuis 2024)
  ┌──────────────────────────────┐
  │  Log Analytics Workspace     │   ← stockage et requêtage KQL
  └──────────────────────────────┘
```

- **Application Insights** est l'**APM** d'Azure : il instrumente votre application (côté serveur et côté client) pour collecter requêtes, dépendances, exceptions, traces et logs custom.
- Depuis 2024, App Insights est **workspace-based** : les données sont en réalité stockées dans un **Log Analytics Workspace (LAW)**, qui se requête avec le langage **KQL** (Kusto Query Language).
- Le **SDK OpenTelemetry** (`azure-monitor-opentelemetry`) est la voie recommandée pour instrumenter le code Python — un standard CNCF, donc transférable.

### Quelques notions

- **Connection string** : ce qu'on colle dans la variable d'environnement `APPLICATIONINSIGHTS_CONNECTION_STRING` pour que le SDK sache où envoyer les données.
- **Sampling** : pour réduire le coût, App Insights échantillonne (par défaut 100 %). Personnalisable.
- **Live Metrics** : un flux temps réel des requêtes / exceptions / dépendances, gratuit, idéal pour debug d'un déploiement.
- **Application Map** : graphe auto-généré de votre app et de ses dépendances (SQL, HTTP, Service Bus, etc.).

## 2. Reprendre les variables du notebook 01

On se branche sur la même Web App. Remettez ici **le nom de votre Resource Group** (convention `rg-stage-<votre-identifiant>`, sur lequel vous êtes Owner). Le notebook en déduit automatiquement l'identifiant, la région et les noms des ressources déjà créées par le notebook 01.

In [ ]:
import os, re, subprocess

# 👇 SEULE VARIABLE À ÉDITER : le nom de votre Resource Group (le même qu'au notebook 01).
#    Convention équipe : rg-stage-<votre-identifiant>
RG = "rg-stage-<votre-identifiant>"

# On dérive l'identifiant et donc tous les autres noms.
m = re.match(r"^rg-stage-(?P<id>[a-z0-9\-]+)$", RG)
assert m, f"❌ Le RG '{RG}' ne suit pas la convention 'rg-stage-<id>'."
USER_ID = m.group("id")

# La région est lue depuis le RG.
LOCATION = subprocess.check_output(
    f"az group show --name {RG} --query location -o tsv", shell=True
).decode().strip()
assert LOCATION, "Impossible de lire la région du RG — vérifiez son nom et vos droits."

WEBAPP     = f"web-stage-{USER_ID}"
APPINSIGHT = f"appi-stage-{USER_ID}"
LAW        = f"law-stage-{USER_ID}"

for k, v in {"LOCATION": LOCATION, "RG": RG, "USER_ID": USER_ID, "WEBAPP": WEBAPP,
             "APPINSIGHT": APPINSIGHT, "LAW": LAW}.items():
    os.environ[k] = v

print(f"Resource Group : {RG}  (région : {LOCATION})")
print(f"Identifiant    : {USER_ID}")
print(f"Web App        : {WEBAPP}")
print(f"App Insights   : {APPINSIGHT}")
print(f"Log Analytics  : {LAW}")

In [ ]:
# Vérifier que le Resource Group existe et que la Web App tourne
!az group show --name $RG --query "{name:name, location:location}" -o table
!az webapp show --name $WEBAPP --resource-group $RG --query "{name:name, state:state, host:defaultHostName}" -o table

## 3. Installer l'extension `az` pour Application Insights

Les commandes `az monitor app-insights *` viennent d'une **extension** de l'`az cli`. On l'installe (ou on la met à jour).

In [ ]:
!az extension add --name application-insights --upgrade --only-show-errors
!az extension show --name application-insights --query "{name:name, version:version}" -o table

## 4. Créer le Log Analytics Workspace

C'est le **stockage sous-jacent** : tous les événements App Insights y atterriront, et c'est lui qu'on requête en KQL.

In [ ]:
!az monitor log-analytics workspace create \
    --resource-group $RG \
    --workspace-name $LAW \
    --location $LOCATION \
    --output table

In [ ]:
# Récupérer l'ID complet du workspace (requis par App Insights)
import subprocess
LAW_ID = subprocess.check_output(
    f'az monitor log-analytics workspace show --resource-group {RG} --workspace-name {LAW} --query id -o tsv',
    shell=True,
).decode().strip()
os.environ["LAW_ID"] = LAW_ID
print(LAW_ID)

## 5. Créer la ressource Application Insights

On lie App Insights au workspace via le paramètre `--workspace`. Sans ce paramètre, on crée une ressource « classic » obsolète : à éviter.

In [ ]:
!az monitor app-insights component create \
    --app $APPINSIGHT \
    --location $LOCATION \
    --resource-group $RG \
    --workspace $LAW_ID \
    --output table

## 6. Brancher la Web App sur Application Insights

Deux choses à faire :

1. Pousser la **connection string** comme variable d'environnement de la Web App (`APPLICATIONINSIGHTS_CONNECTION_STRING`).
2. Activer l'**auto-instrumentation** côté App Service (Linux) pour Python — pas besoin de modifier votre code.

> Pour les apps **.NET, Java, Node.js**, l'auto-instrumentation existe aussi et se configure avec les mêmes app settings (`ApplicationInsightsAgent_EXTENSION_VERSION`, etc.). Voir https://learn.microsoft.com/azure/azure-monitor/app/codeless-overview

In [ ]:
# 6.1 Récupérer la connection string
CONN = subprocess.check_output(
    f'az monitor app-insights component show --app {APPINSIGHT} --resource-group {RG} --query connectionString -o tsv',
    shell=True,
).decode().strip()
os.environ["CONN"] = CONN
print("Connection string récupérée ✓")

In [ ]:
# 6.2 Pousser la connection string + activer l'auto-instrumentation sur la Web App
!az webapp config appsettings set \
    --name $WEBAPP --resource-group $RG \
    --settings \
        APPLICATIONINSIGHTS_CONNECTION_STRING="$CONN" \
        ApplicationInsightsAgent_EXTENSION_VERSION="~3" \
        XDT_MicrosoftApplicationInsights_Mode="recommended" \
    --output table

In [ ]:
# 6.3 Redémarrer la Web App pour prendre en compte les nouveaux settings
!az webapp restart --name $WEBAPP --resource-group $RG

## 7. Générer un peu de trafic

App Insights n'affiche rien tant qu'il n'y a pas eu d'événements. On va « pinger » la Web App une dizaine de fois pour générer des requêtes HTTP que la télémétrie capturera.

In [ ]:
import urllib.request, time

URL = f"https://{WEBAPP}.azurewebsites.net/"
print(f"Ping → {URL}")
for i in range(10):
    try:
        with urllib.request.urlopen(URL, timeout=10) as r:
            print(f"  [{i+1:02d}] {r.status}")
    except Exception as e:
        print(f"  [{i+1:02d}] error: {e}")
    time.sleep(1)

## 8. Envoyer une trace custom depuis Python (OpenTelemetry)

L'auto-instrumentation capture déjà les requêtes HTTP. Mais on veut souvent ajouter du **contexte métier** : durée d'une opération précise, événement custom, etc. C'est le rôle du SDK **`azure-monitor-opentelemetry`**.

Ici on instrumente **ce notebook lui-même** (rôle `notebook-stagiaire`), juste pour démontrer le mécanisme.

In [ ]:
from azure.monitor.opentelemetry import configure_azure_monitor
from opentelemetry import trace
import logging, time

# Configure OpenTelemetry → Azure Monitor (utilise la connection string)
configure_azure_monitor(
    connection_string=CONN,
    resource_attributes={"service.name": "notebook-stagiaire"},
)

tracer = trace.get_tracer("stagiaire.demo")
logger = logging.getLogger("stagiaire.demo")
logger.setLevel(logging.INFO)

# Émet une trace et un log
with tracer.start_as_current_span("calcul-important") as span:
    span.set_attribute("user.id", "stagiaire-42")
    span.set_attribute("feature", "demo-app-insights")
    logger.info("Démarrage du calcul important")
    time.sleep(0.5)  # simule un traitement
    logger.info("Calcul terminé")

print("✓ Trace + log envoyés. Comptez ~1-2 min pour les voir dans App Insights.")

## 9. Explorer les données — premières requêtes KQL

**KQL** (Kusto Query Language) est le langage de requêtage de Log Analytics. Quelques tables utiles d'App Insights :

| Table | Contenu |
|-------|---------|
| `requests` | Requêtes HTTP reçues par votre app |
| `dependencies` | Appels sortants (SQL, HTTP, etc.) |
| `exceptions` | Erreurs non gérées |
| `traces` | Logs et messages custom |
| `customEvents` | Événements métier (clic, conversion…) |
| `customMetrics` | Métriques personnalisées |

On exécute les requêtes via `az monitor app-insights query`.

> ⏱️ Les données mettent en général **1 à 3 minutes** à apparaître après émission. Si une requête renvoie vide, ré-essayez après une courte pause.

In [ ]:
# 9.1 Combien de requêtes la webapp a-t-elle reçues sur les 30 dernières minutes ?
!az monitor app-insights query \
    --app $APPINSIGHT --resource-group $RG \
    --analytics-query "requests | where timestamp > ago(30m) | summarize total=count(), failed=countif(success == false) by bin(timestamp, 5m) | order by timestamp asc" \
    --output table

In [ ]:
# 9.2 Top 5 des routes les plus lentes (p95)
!az monitor app-insights query \
    --app $APPINSIGHT --resource-group $RG \
    --analytics-query "requests | where timestamp > ago(30m) | summarize p95=percentile(duration, 95), count() by name | top 5 by p95 desc" \
    --output table

In [ ]:
# 9.3 Retrouver la trace custom envoyée à la section 8
!az monitor app-insights query \
    --app $APPINSIGHT --resource-group $RG \
    --analytics-query "dependencies | where cloud_RoleName == 'notebook-stagiaire' and timestamp > ago(30m) | project timestamp, name, duration, customDimensions | order by timestamp desc | take 10" \
    --output table

### Quelques requêtes KQL classiques à connaître

```kusto
// Taux d'échec par minute
requests
| where timestamp > ago(1h)
| summarize total=count(), failed=countif(success == false) by bin(timestamp, 1m)
| extend failure_rate = 100.0 * failed / total

// Top exceptions
exceptions
| where timestamp > ago(24h)
| summarize count() by problemId, type
| top 10 by count_ desc

// Map des dépendances : qui appelle quoi
dependencies
| where timestamp > ago(1h)
| summarize count(), avg(duration) by target, name, type
| order by count_ desc

// Logs custom de ce notebook
traces
| where cloud_RoleName == "notebook-stagiaire"
| order by timestamp desc
```

## 11. 🧹 Nettoyage

Le **Resource Group ne vous appartient pas** (il a été créé par votre équipe). **Ne le supprimez pas**.

Pour ne supprimer que ce que **ce notebook** a créé (LAW + App Insights), sans toucher au reste :

## 11. 🧹 Nettoyage

Si vous voulez seulement supprimer ce qu'ajoute **ce notebook** (LAW + App Insights) sans toucher au reste :

In [ ]:
# Suppression ciblée — décommentez pour exécuter.
# !az monitor app-insights component delete --app $APPINSIGHT --resource-group $RG
# !az monitor log-analytics workspace delete --workspace-name $LAW --resource-group $RG --yes --force true
# # Et retirer les app settings côté Web App :
# !az webapp config appsettings delete --name $WEBAPP --resource-group $RG \
#     --setting-names APPLICATIONINSIGHTS_CONNECTION_STRING ApplicationInsightsAgent_EXTENSION_VERSION XDT_MicrosoftApplicationInsights_Mode

print("⚠️  Ne supprimez PAS le Resource Group : il appartient à votre équipe.")
print(f"    RG protégé : {RG}")

## Récap

Vous savez maintenant :
- les **3 piliers de l'observabilité** (logs, métriques, traces) et la stack Azure Monitor / Log Analytics / Application Insights ;
- créer un **Log Analytics Workspace** + une ressource **Application Insights** workspace-based ;
- brancher la **connection string** sur une Web App + activer l'**auto-instrumentation** Python ;
- envoyer des **traces / logs custom** depuis Python avec **OpenTelemetry** ;
- requêter les données avec **KQL** ;
- nettoyer **vos** ressources sans toucher au Resource Group de l'équipe.

**À venir dans ce notebook :**
- **Évaluation des agents Foundry** : groundedness, relevance, fluency, safety, jeux de données d'évaluation, evaluators built-in et custom.

📚 Pour aller plus loin :
- Doc App Insights : https://learn.microsoft.com/azure/azure-monitor/app/app-insights-overview
- OpenTelemetry pour Azure Monitor (Python) : https://learn.microsoft.com/azure/azure-monitor/app/opentelemetry-enable?tabs=python
- Apprendre KQL : https://learn.microsoft.com/training/paths/kusto-query-language/
- Évaluations Foundry : https://learn.microsoft.com/azure/foundry/how-to/develop/evaluate-sdk

## Prochaines étapes

Vous arrivez au **bout du parcours initial** 🎉. Le tableau de marche :

- ✅ **01** — Fondamentaux Azure & PaaS
- ✅ **02** — Agent IA + Workflow Foundry
- ✅ **03** — Projet, DevOps & Agile
- ✅ **04** — Architecture web
- ✅ **05** — Sécurité cloud
- ✅ **06** — Monitoring & évaluation *(vous y êtes)*

**Pour aller plus loin :**
- Brancher le **tracing automatique des agents Foundry** vers cet App Insights (notebook à venir).
- Compléter ce notebook avec une **section évaluation** : groundedness, relevance, fluency, safety, jeux d'évaluations Foundry.
- Mettre en place une **Managed Identity** App Service → Foundry (cf. notebook 05 §4).
- Industrialiser tout ça en **Bicep / Terraform** (Infrastructure as Code).

📚 Pour aller plus loin :
- Doc App Insights : https://learn.microsoft.com/azure/azure-monitor/app/app-insights-overview
- OpenTelemetry pour Azure Monitor (Python) : https://learn.microsoft.com/azure/azure-monitor/app/opentelemetry-enable?tabs=python
- Apprendre KQL : https://learn.microsoft.com/training/paths/kusto-query-language/
- Évaluations Foundry : https://learn.microsoft.com/azure/foundry/how-to/develop/evaluate-sdk
- Tracing agents Foundry : https://learn.microsoft.com/azure/foundry/observability/concepts/trace-agent-concept